# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

### This is an AI assistant that lets you input a technical question and receive an answer. You can choose to use either a Llama-based model or OpenAI’s model for your query.

Note: Make sure that Ollama is installed and running on your machine.

In [ ]:
# imports
import os
from dotenv import load_dotenv
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
import ollama
from IPython.display import display, Markdown, update_display

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
#check api key
if api_key and api_key.startswith("sk-proj-") and len(api_key)>10:
    print("API key looks good so far")
    openai = OpenAI()
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

openai = OpenAI()

In [ ]:
# Get gpt-4o-mini to answer, with streaming
#def answer_with_streaming(question)
def answer_technical_questions(llama = False):
    '''
    The user types in a technical question and it is answered using llma or openai
    
    args:
        llama bool, wheather to use llama or not, if False use openai 
        
    '''
    try:
        #prompt user to ask a question, if it is left empty ask user to enter a question
        question = input('Please ask me a question: ')
        if not question:
            print('You have to type a technical question')
        
        #create system and user prompts
        system_prompt = "You are a technical helper that help explain technical questions in an easy to understand terms. Please provide\
                    details. Please respond in markdown "
    
        user_prompt = "You receive a technical question;"
        user_prompt += question
        user_prompt += "answer this question in easy to understand terms and give details"
        user_prompt = user_prompt[:5_000]

        #define messages
        messages = [
            {"role":"system","content":system_prompt},
            {"role":"user","content":user_prompt}
            ]

        display_handle = display(Markdown(""),display_id = True)

        #if llama = True then run llama model
        if llama:
            stream_llama = ollama.chat(
            model = MODEL_LLAMA,
            messages =messages,
            stream = True #streaming is on
            )
            
            response = ''
            #loop over chunks in the stream and display them in markdown
            for chunk in stream_llama:
                response += chunk['message']['content']
                update_display(Markdown(response), display_id=display_handle.display_id)
                
        else:
            #if llama = False then run openai
            stream = openai.chat.completions.create(
            model = MODEL_GPT,
            messages = messages,
            stream = True
            )
    
            response = ''
            for chunk in stream:
                response += chunk.choices[0].delta.content or ''  
                update_display(Markdown(response), display_id = display_handle.display_id)
    except Exception as e:
        print(f'Error: {e}')

In [ ]:
answer_technical_questions()